In [1]:
!pip install albumentations
!pip install opencv-python

# 0. Library Importation

In [13]:
import os
import random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset
import torchvision
import albumentations as A

In [3]:
print("All libraries imported successfully")
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("OpenCV version:", cv2.__version__)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Please enable GPU in Kaggle settings.")

!nvidia-smi

All libraries imported successfully
PyTorch version: 2.10.0+cpu
Torchvision version: 0.25.0+cpu
OpenCV version: 4.13.0
CUDA available: False
Device: cpu
GPU is not available. Please enable GPU in Kaggle settings.
/bin/bash: line 1: nvidia-smi: command not found


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 1. Dataset Load

In [5]:
# ===== PATH =====
BASE_PATH = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE"

train_img_dir = os.path.join(BASE_PATH, "training/images")
train_mask_dir = os.path.join(BASE_PATH, "training/1st_manual")

test_img_dir = os.path.join(BASE_PATH, "test/images")
test_mask_dir = os.path.join(BASE_PATH, "test/1st_manual")


In [6]:
# ===== LOAD TRAIN =====
train_images = sorted(os.listdir(train_img_dir))

train_image_paths = []
train_mask_paths = []

for img_name in train_images:
    img_id = img_name.split('_')[0]
    mask_name = f"{img_id}_manual1.gif"
    
    train_image_paths.append(os.path.join(train_img_dir, img_name))
    train_mask_paths.append(os.path.join(train_mask_dir, mask_name))

In [7]:
# ===== LOAD TEST =====
test_images = sorted(os.listdir(test_img_dir))

test_image_paths = []
test_mask_paths = []

for img_name in test_images:
    img_id = img_name.split('_')[0]
    mask_name = f"{img_id}_manual1.gif"
    
    test_image_paths.append(os.path.join(test_img_dir, img_name))
    test_mask_paths.append(os.path.join(test_mask_dir, mask_name))

In [ ]:
class DRIVEDataset(Dataset):
    def __init__(self, image_paths, mask_paths, img_size=(256, 256)):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load img & mask
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Convert("RGB") & Convert("L") for gray mask 
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        img = img.resize(self.img_size) # (256, 256)
        mask = mask.resize(self.img_size, Image.NEAREST) 

        img = np.array(img, dtype=np.float32)
        mask = np.array(mask, dtype=np.float32)

        img = img / 255.0 # Normalize

        # Convert mask 0/1
        mask = mask / 255.0 

        # Swap axis
        img = np.transpose(img, (2, 0, 1))
        
        # Add channel axis for mask to become (1, H, W)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [15]:
# ===== TEST CLASS DATASET =====
train_dataset = DRIVEDataset(train_image_paths, train_mask_paths)

sample_img, sample_mask = train_dataset[0]

print("Dataset check")
print(f"Image tensor shape: {sample_img.shape}")
print(f"Image max/min: {sample_img.max().item()} / {sample_img.min().item()}")

print(f"Mask tensor shape: {sample_mask.shape}")
print(f"Mask unique values: {torch.unique(sample_mask).tolist()}")

Dataset check
Image tensor shape: torch.Size([3, 256, 256])
Image max/min: 1.0 / 0.0
Mask tensor shape: torch.Size([1, 256, 256])
Mask unique values: [0.0, 1.0]
